In [8]:
import sys, os
sys.path.append(r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\src")
from BSM import bsm_price, bsm_implied_vol, put_call_parity_check

def test_known_case():
    # S=100, K=100, T=1, sigma=0.2, r=0.05, q=0 → call ≈ 10.4506
    p = bsm_price(100, 100, 1, 0.2, r=0.05, q=0.0, kind="call")
    assert abs(p - 10.4506) < 1e-3, p
    print("known case OK:", round(p, 4))

def test_parity():
    assert put_call_parity_check(100, 100, 0.5, 0.25, r=0.04, q=0.013)
    print("put-call parity OK")

def test_roundtrip_iv():
    iv = bsm_implied_vol(10.4506, 100, 100, 1, r=0.05, q=0.0, kind="call")
    assert abs(iv - 0.2) < 1e-4, iv
    print("IV roundtrip OK:", round(iv, 4))

def test_expiry_intrinsic():
    assert bsm_price(110, 100, 0, 0.2, kind="call") == 10.0
    assert bsm_price(90, 100, 0, 0.2, kind="put") == 10.0
    print("expiry intrinsic OK")

if __name__ == "__main__":
    test_known_case()
    test_parity()
    test_roundtrip_iv()
    test_expiry_intrinsic()
    print("ALL BSM TESTS PASSED")

known case OK: 10.4506
put-call parity OK
IV roundtrip OK: 0.2
expiry intrinsic OK
ALL BSM TESTS PASSED


In [10]:
# src/bsm_eval.py
import os, numpy as np, pandas as pd, pyarrow.parquet as pq, lightgbm as lgb, sys
sys.path.append(r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\src")
from bsm import bsm_price

CLEAN_DIR = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\clean"
MODEL     = os.path.join(CLEAN_DIR, "lgbm_iv.txt")
R, Q      = 0.04, 0.013
TTE_MAX   = 7
BATCH     = 200_000

FEATS = ["implied_volatility","iv_lag1","iv_change","iv_ma_5","iv_std_5",
         "mid","rel_spread","bid_size","ask_size","volume","open_interest",
         "delta","gamma","theta","vega","rho","strike","tte_days","moneyness",
         "spy_ret_1d","spy_rv_20d","type","in_the_money"]

model = lgb.Booster(model_file=MODEL)

def prep(df):
    for c in FEATS:
        df[c] = df[c].astype("category") if c == "type" else pd.to_numeric(df[c], errors="coerce").astype("float32")
    return df

def eval_split(name):
    f = os.path.join(CLEAN_DIR, f"{name}_features.parquet")
    pf = pq.ParquetFile(f)
    cols = FEATS + ["target_iv","spy_close","strike","tte_days","type","mid","bid","ask","expiration","date","contract_id"]

    abs_sum = sq_sum = 0.0
    n = 0
    inside = 0
    per_tte = {}

    for b in pf.iter_batches(batch_size=BATCH, columns=cols):
        df = prep(b.to_pandas())
        df = df[(df["target_iv"].notna()) & (df["tte_days"] >= 0) & (df["tte_days"] <= TTE_MAX)]
        if len(df) == 0: continue

        # predicted IV
        iv_hat = model.predict(df[FEATS], num_iteration=model.best_iteration).astype("float64")
        iv_hat = np.clip(iv_hat, 1e-3, 5.0)          # guard
        S  = df["spy_close"].astype("float64").values
        K  = df["strike"].astype("float64").values
        T  = (df["tte_days"].astype("float64").values + 1e-3) / 365.0   # avoid T=0
        T  = np.maximum(T, 1.0/365.0)
        kind = df["type"].astype(str).values

        # BSM theoretical prices (vectorised)
        d1 = (np.log(S/K) + (R - Q + 0.5*iv_hat**2)*T) / (iv_hat*np.sqrt(T))
        d2 = d1 - iv_hat*np.sqrt(T)
        from scipy.stats import norm
        call = S*np.exp(-Q*T)*norm.cdf(d1) - K*np.exp(-R*T)*norm.cdf(d2)
        put  = K*np.exp(-R*T)*norm.cdf(-d2) - S*np.exp(-Q*T)*norm.cdf(-d1)
        price_hat = np.where(kind == "call", call, put)

        y      = df["mid"].astype("float64").values
        bid    = df["bid"].astype("float64").values
        ask    = df["ask"].astype("float64").values
        ok     = np.isfinite(price_hat) & np.isfinite(y) & (y > 0)
        if ok.sum() == 0: continue

        err  = price_hat[ok] - y[ok]
        abs_sum += np.abs(err).sum()
        sq_sum  += (err**2).sum()
        n       += ok.sum()
        inside  += ((price_hat[ok] >= bid[ok]) & (price_hat[ok] <= ask[ok])).sum()

        # per TTE bucket
        tte_b = df["tte_days"].values[ok]
        for v, e in zip(tte_b, np.abs(err)):
            per_tte.setdefault(int(v), []).append(e)

    print(f"\n=== {name} (TTE 0-{TTE_MAX}) ===")
    print(f"n={n:,}  price MAE={abs_sum/n:.4f}  RMSE={np.sqrt(sq_sum/n):.4f}  inside_bidask={inside/n:.2%}")
    for k in sorted(per_tte):
        arr = np.array(per_tte[k])
        print(f"  TTE={k}: n={len(arr):>8,}  MAE={arr.mean():.4f}")

for s in ["val","test"]:
    eval_split(s)


=== val (TTE 0-7) ===
n=636,566  price MAE=0.1903  RMSE=0.3302  inside_bidask=24.84%
  TTE=1: n= 107,972  MAE=0.2013
  TTE=2: n=  81,596  MAE=0.2114
  TTE=3: n=  79,966  MAE=0.1791
  TTE=4: n=  78,220  MAE=0.1683
  TTE=5: n=  71,652  MAE=0.1934
  TTE=6: n=  94,230  MAE=0.1763
  TTE=7: n= 122,930  MAE=0.1969

=== test (TTE 0-7) ===
n=340,342  price MAE=0.3095  RMSE=0.6311  inside_bidask=37.56%
  TTE=1: n=  57,230  MAE=0.3639
  TTE=2: n=  44,620  MAE=0.3106
  TTE=3: n=  42,904  MAE=0.2541
  TTE=4: n=  41,830  MAE=0.2792
  TTE=5: n=  38,036  MAE=0.3317
  TTE=6: n=  50,400  MAE=0.2989
  TTE=7: n=  65,322  MAE=0.3123


In [11]:
# src/bsm_validate.py
import os, numpy as np, pandas as pd, pyarrow.parquet as pq
from scipy.stats import norm

CLEAN_DIR = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\clean"
R, Q    = 0.04, 0.013
TTE_MAX = 7
BATCH   = 200_000

def eval_split(name):
    f = os.path.join(CLEAN_DIR, f"{name}_features.parquet")
    pf = pq.ParquetFile(f)
    cols = ["implied_volatility","spy_close","strike","tte_days","type",
            "mid","bid","ask","rel_spread"]

    abs_sum = sq_sum = 0.0
    n = inside = 0
    per_tte = {}
    per_type = {}

    for b in pf.iter_batches(batch_size=BATCH, columns=cols):
        df = b.to_pandas()
        df = df[(df["implied_volatility"].notna()) &
                (df["tte_days"] >= 1) & (df["tte_days"] <= TTE_MAX) &   # skip TTE=0
                (df["mid"] > 0) & (df["bid"] > 0) & (df["ask"] > 0) &
                (df["ask"] >= df["bid"])]
        if len(df) == 0: continue

        S    = df["spy_close"].astype("float64").values
        K    = df["strike"].astype("float64").values
        sig  = df["implied_volatility"].astype("float64").values
        T    = df["tte_days"].astype("float64").values / 365.0
        kind = df["type"].astype(str).values

        d1 = (np.log(S/K) + (R - Q + 0.5*sig**2)*T) / (sig*np.sqrt(T))
        d2 = d1 - sig*np.sqrt(T)
        call = S*np.exp(-Q*T)*norm.cdf(d1) - K*np.exp(-R*T)*norm.cdf(d2)
        put  = K*np.exp(-R*T)*norm.cdf(-d2) - S*np.exp(-Q*T)*norm.cdf(-d1)
        price_hat = np.where(kind == "call", call, put)

        y   = df["mid"].astype("float64").values
        bid = df["bid"].astype("float64").values
        ask = df["ask"].astype("float64").values

        ok = np.isfinite(price_hat)
        err = price_hat[ok] - y[ok]
        abs_sum += np.abs(err).sum()
        sq_sum  += (err**2).sum()
        n       += ok.sum()
        inside  += ((price_hat[ok] >= bid[ok]) & (price_hat[ok] <= ask[ok])).sum()

        # bucket by TTE
        for v, e in zip(df["tte_days"].values[ok], np.abs(err)):
            per_tte.setdefault(int(v), []).append(e)
        # bucket by type
        for v, e in zip(kind[ok], np.abs(err)):
            per_type.setdefault(v, []).append(e)

    print(f"\n=== {name} (BSM on ACTUAL today's IV, TTE 1-{TTE_MAX}) ===")
    print(f"n={n:,}  price MAE={abs_sum/n:.4f}  RMSE={np.sqrt(sq_sum/n):.4f}  inside_bidask={inside/n:.2%}")
    print("By type:")
    for k, v in per_type.items():
        print(f"  {k}: n={len(v):>8,}  MAE={np.mean(v):.4f}")
    print("By TTE:")
    for k in sorted(per_tte):
        arr = np.array(per_tte[k])
        print(f"  TTE={k}: n={len(arr):>8,}  MAE={arr.mean():.4f}")

for s in ["val","test"]:
    eval_split(s)


=== val (BSM on ACTUAL today's IV, TTE 1-7) ===
n=517,285  price MAE=0.1036  RMSE=0.1934  inside_bidask=49.12%
By type:
  put: n= 270,074  MAE=0.0788
  call: n= 247,211  MAE=0.1308
By TTE:
  TTE=1: n=  74,390  MAE=0.1018
  TTE=2: n=  63,084  MAE=0.0786
  TTE=3: n=  62,822  MAE=0.0854
  TTE=4: n=  63,723  MAE=0.1006
  TTE=5: n=  61,274  MAE=0.1090
  TTE=6: n=  83,188  MAE=0.1129
  TTE=7: n= 108,804  MAE=0.1216

=== test (BSM on ACTUAL today's IV, TTE 1-7) ===
n=284,095  price MAE=0.1393  RMSE=0.4099  inside_bidask=61.71%
By type:
  call: n= 142,854  MAE=0.1661
  put: n= 141,241  MAE=0.1122
By TTE:
  TTE=1: n=  42,194  MAE=0.1637
  TTE=2: n=  35,495  MAE=0.1337
  TTE=3: n=  34,908  MAE=0.1120
  TTE=4: n=  34,988  MAE=0.1273
  TTE=5: n=  32,755  MAE=0.1497
  TTE=6: n=  44,811  MAE=0.1420
  TTE=7: n=  58,944  MAE=0.1406


In [13]:
import os, duckdb, numpy as np, pandas as pd, lightgbm as lgb
from scipy.stats import norm

CLEAN_DIR = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\clean"
SPY_FILE  = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\underlying_prices.parquet"
TEMP_DIR  = r"D:\temp\duckdb"
MODEL     = os.path.join(CLEAN_DIR, "lgbm_iv.txt")
R, Q      = 0.04, 0.013
TTE_MAX   = 7
BATCH     = 200_000

FEATS = ["implied_volatility","iv_lag1","iv_change","iv_ma_5","iv_std_5",
         "mid","rel_spread","bid_size","ask_size","volume","open_interest",
         "delta","gamma","theta","vega","rho","strike","tte_days","moneyness",
         "spy_ret_1d","spy_rv_20d","type","in_the_money"]

con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute(f"SET temp_directory='{TEMP_DIR}'")
con.execute("SET threads=4")

model = lgb.Booster(model_file=MODEL)
spy = SPY_FILE.replace("\\","/")

def bsm_vec(S, K, T, sig, kind):
    d1 = (np.log(S/K) + (R - Q + 0.5*sig**2)*T) / (sig*np.sqrt(T))
    d2 = d1 - sig*np.sqrt(T)
    call = S*np.exp(-Q*T)*norm.cdf(d1) - K*np.exp(-R*T)*norm.cdf(d2)
    put  = K*np.exp(-R*T)*norm.cdf(-d2) - S*np.exp(-Q*T)*norm.cdf(-d1)
    return np.where(kind=="call", call, put)

def eval_split(name):
    src = os.path.join(CLEAN_DIR, f"{name}_features.parquet").replace("\\","/")
    q = f"""
        WITH feat AS (
            SELECT *,
                LEAD(mid) OVER w AS next_mid,
                LEAD(bid) OVER w AS next_bid,
                LEAD(ask) OVER w AS next_ask,
                LEAD(date) OVER w AS next_date
            FROM read_parquet('{src}')
            WINDOW w AS (PARTITION BY contract_id ORDER BY date)
        )
        SELECT f.contract_id, f.date, f.strike, f.type, f.expiration,
            f.implied_volatility, f.iv_lag1, f.iv_change, f.iv_ma_5, f.iv_std_5,
            f.mid, f.rel_spread, f.bid_size, f.ask_size, f.volume, f.open_interest,
            f.delta, f.gamma, f.theta, f.vega, f.rho, f.tte_days,
            f.moneyness, f.spy_ret_1d, f.spy_rv_20d, f.in_the_money,
            f.next_mid, f.next_bid, f.next_ask,
            CAST(f.next_date AS DATE) - CAST(f.date AS DATE) AS days_to_next,
            s.close AS next_spy_close
        FROM feat f
        LEFT JOIN '{spy}' s ON CAST(f.next_date AS DATE) = CAST(s.date AS DATE)
        WHERE f.next_mid IS NOT NULL AND f.next_mid > 0
          AND CAST(f.next_date AS DATE) - CAST(f.date AS DATE) BETWEEN 1 AND 4
    """
    res = {"ml": [0.,0.,0,0], "pers": [0.,0.,0,0], "mid_pers": [0.,0.,0,0]}

    cur = con.execute(q)
    while True:
        df = cur.fetch_df_chunk(BATCH)
        if df is None or len(df) == 0:
            break
        df = df[(df["tte_days"] >= 0) & (df["tte_days"] <= TTE_MAX)
                & df["next_spy_close"].notna() & (df["next_spy_close"] > 0)]
        if len(df) == 0: continue
        for c in FEATS:
            df[c] = df[c].astype("category") if c == "type" else pd.to_numeric(df[c], errors="coerce").astype("float32")

        iv_ml   = np.clip(model.predict(df[FEATS], num_iteration=model.best_iteration).astype("float64"), 1e-3, 5.0)
        iv_pers = np.clip(df["implied_volatility"].astype("float64").values, 1e-3, 5.0)

        S = df["next_spy_close"].astype("float64").values
        K = df["strike"].astype("float64").values
        days = df["days_to_next"].astype("float64").values
        T = np.maximum((df["tte_days"].astype("float64").values - days), 0.0) / 365.0
        T = np.maximum(T, 1.0/365.0)
        kind = df["type"].astype(str).values

        p_ml   = bsm_vec(S, K, T, iv_ml, kind)
        p_pers = bsm_vec(S, K, T, iv_pers, kind)

        y        = df["next_mid"].astype("float64").values
        bid      = df["next_bid"].astype("float64").values
        ask      = df["next_ask"].astype("float64").values
        today_mid = df["mid"].astype("float64").values
        base_ok = np.isfinite(y) & (y>0) & np.isfinite(bid) & np.isfinite(ask)

        for label, pred in [("ml", p_ml), ("pers", p_pers), ("mid_pers", today_mid)]:
            ok = base_ok & np.isfinite(pred)
            if ok.sum() == 0: continue
            e = pred[ok] - y[ok]
            res[label][0] += np.abs(e).sum()
            res[label][1] += (e**2).sum()
            res[label][2] += ok.sum()
            res[label][3] += ((pred[ok]>=bid[ok]) & (pred[ok]<=ask[ok])).sum()

    print(f"\n=== {name} — next-day BSM vs next-day mid (TTE today 0-{TTE_MAX}, gap 1-4d) ===")
    for label,(a,sq,n,ins) in res.items():
        if n == 0: continue
        print(f"  {label:9s} n={n:>9,}  MAE={a/n:.4f}  RMSE={np.sqrt(sq/n):.4f}  inside={ins/n:.2%}")

for s in ["val","test"]:
    eval_split(s)


=== val — next-day BSM vs next-day mid (TTE today 0-7, gap 1-4d) ===
  ml        n=  499,492  MAE=0.1949  RMSE=0.3659  inside=33.63%
  pers      n=  499,492  MAE=0.1977  RMSE=0.3868  inside=41.04%
  mid_pers  n=  499,492  MAE=1.5029  RMSE=2.6792  inside=22.85%

=== test — next-day BSM vs next-day mid (TTE today 0-7, gap 1-4d) ===
  ml        n=  266,576  MAE=0.3393  RMSE=0.7158  inside=44.51%
  pers      n=  266,576  MAE=0.3427  RMSE=0.7559  inside=51.27%
  mid_pers  n=  266,576  MAE=2.3868  RMSE=4.9892  inside=24.14%


In [14]:
# src/sabr.py
import numpy as np
from scipy.optimize import least_squares

BETA = 0.5  # common default

def sabr_iv(F, K, T, alpha, beta, rho, nu):
    """Hagan 2002 lognormal SABR implied vol (approximation)."""
    K = np.asarray(K, dtype=float)
    if T <= 0 or alpha <= 0 or nu <= 0:
        return np.full_like(K, np.nan)
    eps = 1e-8
    K = np.maximum(K, eps)
    FK = F * K
    logFK = np.log(F / K)
    FK_b = FK ** ((1 - beta) / 2.0)
    z = (nu / alpha) * FK_b * logFK
    # handle ATM z=0
    z = np.where(np.abs(z) < 1e-8, 1e-8, z)
    x = np.log((np.sqrt(1 - 2*rho*z + z*z) + z - rho) / (1 - rho))
    A = alpha / (FK_b * (1 + ((1-beta)**2/24)*logFK**2 + ((1-beta)**4/1920)*logFK**4))
    B = 1 + (((1-beta)**2/24)*alpha**2/FK**(1-beta)
             + 0.25*rho*beta*nu*alpha/FK_b
             + ((2-3*rho**2)/24)*nu**2) * T
    return A * (z / x) * B

def calibrate_smile(F, K, iv, T, beta=BETA):
    """Fit alpha, rho, nu to observed IVs at strikes K."""
    K = np.asarray(K, float); iv = np.asarray(iv, float)
    ok = np.isfinite(iv) & (iv > 0) & (iv < 5) & (K > 0)
    K, iv = K[ok], iv[ok]
    if len(K) < 4:
        return None
    atm_iv = np.interp(F, K, iv)
    x0 = np.array([atm_iv, 0.0, 0.3])
    bounds = ([1e-3, -0.999, 1e-3], [5.0, 0.999, 5.0])
    def resid(p):
        return sabr_iv(F, K, T, p[0], beta, p[1], p[2]) - iv
    try:
        r = least_squares(resid, x0, bounds=bounds, max_nfev=200)
    except Exception:
        return None
    if not r.success:
        return None
    return {"alpha": r.x[0], "rho": r.x[1], "nu": r.x[2],
            "rmse": float(np.sqrt(np.mean(r.fun**2))), "n": len(K)}

In [15]:
# src/sabr_eval.py
import os, duckdb, numpy as np, pandas as pd, sys
# sys.path.append(os.path.dirname(__file__))
from sabr import sabr_iv, calibrate_smile

CLEAN_DIR = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\clean"
SPY_FILE  = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\underlying_prices.parquet"
TEMP_DIR  = r"D:\temp\duckdb"
TTE_MAX   = 30

con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute(f"SET temp_directory='{TEMP_DIR}'")
con.execute("SET threads=4")

spy = SPY_FILE.replace("\\","/")
srcs = [os.path.join(CLEAN_DIR,f"options_{y}_clean.parquet").replace("\\","/") for y in (2024,2025)]

# small slice: 10 days per month, TTE 1-30, good quotes
q = f"""
WITH o AS (
    SELECT o.*, CAST(o.date AS DATE) d, CAST(o.expiration AS DATE) e
    FROM read_parquet({srcs}) o
    WHERE implied_volatility BETWEEN 0.01 AND 2.0
      AND bid > 0 AND ask > bid
      AND CAST(o.expiration AS DATE) > CAST(o.date AS DATE)
      AND CAST(o.expiration AS DATE) - CAST(o.date AS DATE) BETWEEN 1 AND {TTE_MAX}
      AND EXTRACT(DAY FROM CAST(o.date AS DATE)) <= 10
)
SELECT o.d date, o.e expiration, o.strike, o.type, o.implied_volatility iv,
       s.close spy_close,
       CAST(o.e AS DATE) - CAST(o.d AS DATE) tte_days
FROM o LEFT JOIN '{spy}' s ON o.d = CAST(s.date AS DATE)
WHERE s.close IS NOT NULL
"""
df = con.execute(q).df()
print("rows:", len(df), "| days:", df["date"].nunique())

rows = []
for (d, e), g in df.groupby(["date","expiration"], sort=False):
    F = float(g["spy_close"].iloc[0]) * np.exp(0.0)  # forward≈spot (q≈0 for short T)
    T = float(g["tte_days"].iloc[0]) / 365.0
    if len(g) < 6: continue
    fit = calibrate_smile(F, g["strike"].values, g["iv"].values, T)
    if fit is None: continue
    fitted = sabr_iv(F, g["strike"].values, T, fit["alpha"], 0.5, fit["rho"], fit["nu"])
    rows.append({"date": d, "expiration": e, "n": fit["n"],
                 "rmse": fit["rmse"], "alpha": fit["alpha"],
                 "rho": fit["rho"], "nu": fit["nu"],
                 "max_err": float(np.nanmax(np.abs(fitted - g["iv"].values)))})

res = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print("\ncalibrated smiles:", len(res))
print(res[["rmse","max_err","n"]].describe().round(5).to_string())
print("\nworst 10 fits:")
print(res.nlargest(10, "rmse")[["date","expiration","n","rmse","max_err"]].to_string(index=False))
res.to_csv("audit_out/sabr_fit_2024_2025.csv", index=False)
print("\nsaved: audit_out/sabr_fit_2024_2025.csv")

rows: 481108 | days: 165

calibrated smiles: 2096
             rmse     max_err           n
count  2096.00000  2096.00000  2096.00000
mean      0.13468     0.45690   229.53626
std       0.09597     0.33626   103.91995
min       0.00282     0.00521    42.00000
25%       0.05901     0.20142   176.00000
50%       0.12343     0.36216   216.00000
75%       0.17438     0.60601   277.00000
max       0.68292     2.37625   716.00000

worst 10 fits:
      date expiration   n     rmse  max_err
2025-04-08 2025-04-09 385 0.682917 1.530550
2025-04-07 2025-04-08 360 0.653019 1.559253
2025-04-08 2025-04-10 439 0.648015 1.538039
2025-04-09 2025-04-10 373 0.618541 1.573801
2025-04-09 2025-04-11 484 0.614366 1.509093
2025-04-07 2025-04-09 382 0.598347 1.400924
2025-04-10 2025-04-11 372 0.578299 1.499397
2025-04-07 2025-04-10 395 0.576456 1.327070
2025-04-08 2025-04-11 540 0.561578 1.221194
2024-04-09 2024-04-10 162 0.559697 1.357013

saved: audit_out/sabr_fit_2024_2025.csv


In [17]:
# src/sabr_eval_v2.py
import os, duckdb, numpy as np, pandas as pd, sys
# sys.path.append(os.path.dirname(__file__))
from sabr import sabr_iv, calibrate_smile

CLEAN_DIR = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\clean"
SPY_FILE  = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\underlying_prices.parquet"
TEMP_DIR  = r"D:\temp\duckdb"
R, Q      = 0.04, 0.013
TTE_MAX   = 30

con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute(f"SET temp_directory='{TEMP_DIR}'")
con.execute("SET threads=4")

spy = SPY_FILE.replace("\\","/")
srcs = [os.path.join(CLEAN_DIR,f"options_{y}_clean.parquet").replace("\\","/") for y in (2024,2025)]

# OTM-only + delta band + good quotes + proper forward
q = f"""
WITH o AS (
    SELECT o.contract_id, CAST(o.date AS DATE) d, CAST(o.expiration AS DATE) e,
           o.strike, o.type, o.implied_volatility iv, o.delta, o.bid, o.ask,
           CASE WHEN o.type='call' THEN o.delta ELSE -o.delta END AS abs_delta,
           CAST(o.expiration AS DATE) - CAST(o.date AS DATE) AS tte_days
    FROM read_parquet({srcs}) o
    WHERE implied_volatility BETWEEN 0.02 AND 2.0
      AND bid > 0 AND ask > bid
      AND CAST(o.expiration AS DATE) > CAST(o.date AS DATE)
      AND CAST(o.expiration AS DATE) - CAST(o.date AS DATE) BETWEEN 1 AND {TTE_MAX}
      AND EXTRACT(DAY FROM CAST(o.date AS DATE)) <= 10
      AND ABS(o.delta) BETWEEN 0.05 AND 0.95
)
SELECT o.d date, o.e expiration, o.strike, o.type, o.iv,
       s.close spy_close, o.tte_days,
       s.close * EXP(({R} - {Q}) * (o.tte_days/365.0)) AS F
FROM o LEFT JOIN '{spy}' s ON o.d = CAST(s.date AS DATE)
WHERE s.close IS NOT NULL
  -- OTM-only: calls above forward, puts below forward
  AND ((o.type='call' AND o.strike >= s.close * EXP(({R}-{Q})*(o.tte_days/365.0)))
    OR (o.type='put'  AND o.strike <= s.close * EXP(({R}-{Q})*(o.tte_days/365.0))))
"""
df = con.execute(q).df()
print("rows:", len(df), "| days:", df["date"].nunique())

rows = []
for (d, e), g in df.groupby(["date","expiration"], sort=False):
    if len(g) < 8: continue
    F = float(g["F"].iloc[0])
    T = float(g["tte_days"].iloc[0]) / 365.0
    fit = calibrate_smile(F, g["strike"].values, g["iv"].values, T)
    if fit is None: continue
    fitted = sabr_iv(F, g["strike"].values, T, fit["alpha"], 0.5, fit["rho"], fit["nu"])
    rows.append({"date": d, "expiration": e, "n": fit["n"],
                 "rmse": fit["rmse"], "alpha": fit["alpha"],
                 "rho": fit["rho"], "nu": fit["nu"],
                 "max_err": float(np.nanmax(np.abs(fitted - g["iv"].values)))})

res = pd.DataFrame(rows)
print("\ncalibrated smiles:", len(res))
if len(res):
    print(res[["rmse","max_err","n"]].describe().round(5).to_string())
    print("\nworst 10 fits:")
    print(res.nlargest(10,"rmse")[["date","expiration","n","rmse","max_err"]].to_string(index=False))
    res.to_csv("audit_out/sabr_fit_v2.csv", index=False)
    print("saved: audit_out/sabr_fit_v2.csv")

rows: 91947 | days: 165

calibrated smiles: 2093
             rmse     max_err           n
count  2093.00000  2093.00000  2093.00000
mean      0.01462     0.02457    43.92117
std       0.04879     0.06279    23.63934
min       0.00124     0.00386     8.00000
25%       0.00314     0.00675    25.00000
50%       0.00401     0.00926    40.00000
75%       0.00664     0.01568    58.00000
max       0.63203     0.71838   167.00000

worst 10 fits:
      date expiration   n     rmse  max_err
2025-04-08 2025-04-09  75 0.632029 0.687226
2025-04-08 2025-04-10 104 0.590891 0.680515
2025-04-07 2025-04-08  68 0.546644 0.718381
2025-04-08 2025-04-11 120 0.534261 0.606736
2025-04-07 2025-04-09  95 0.508351 0.639982
2025-04-07 2025-04-10 113 0.472727 0.581543
2025-04-09 2025-04-10  58 0.441647 0.589273
2025-04-10 2025-04-11  64 0.429213 0.528270
2025-04-07 2025-04-11 122 0.427683 0.534638
2025-04-08 2025-04-14 137 0.366377 0.397659
saved: audit_out/sabr_fit_v2.csv


In [18]:
# src/heston_eval.py
import os, duckdb, numpy as np, pandas as pd, sys
sys.path.append(r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\src")
from heston import calibrate_heston, heston_price

CLEAN_DIR = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\clean"
SPY_FILE  = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\underlying_prices.parquet"
TEMP_DIR  = r"D:\temp\duckdb"
R, Q      = 0.04, 0.013
TTE_MAX   = 30

con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute(f"SET temp_directory='{TEMP_DIR}'")
con.execute("SET threads=4")
spy = SPY_FILE.replace("\\","/")
srcs = [os.path.join(CLEAN_DIR,f"options_{y}_clean.parquet").replace("\\","/") for y in (2024,2025)]

q = f"""
WITH o AS (
    SELECT CAST(o.date AS DATE) d, CAST(o.expiration AS DATE) e, o.strike, o.type,
           o.implied_volatility iv, o.delta,
           CAST(o.expiration AS DATE) - CAST(o.date AS DATE) AS tte_days
    FROM read_parquet({srcs}) o
    WHERE implied_volatility BETWEEN 0.02 AND 2.0
      AND bid > 0 AND ask > bid
      AND CAST(o.expiration AS DATE) > CAST(o.date AS DATE)
      AND CAST(o.expiration AS DATE) - CAST(o.date AS DATE) BETWEEN 7 AND {TTE_MAX}
      AND EXTRACT(DAY FROM CAST(o.date AS DATE)) <= 10
      AND ABS(o.delta) BETWEEN 0.10 AND 0.90
)
SELECT o.d date, o.e expiration, o.strike, o.type, o.iv,
       s.close spy_close, o.tte_days,
       s.close * EXP(({R}-{Q})*(o.tte_days/365.0)) AS F
FROM o LEFT JOIN '{spy}' s ON o.d = CAST(s.date AS DATE)
WHERE s.close IS NOT NULL
"""
df = con.execute(q).df()
print("rows:", len(df), "| days:", df["date"].nunique())

rows = []
for (d, e), g in df.groupby(["date","expiration"], sort=False):
    if len(g) < 8: continue
    S = float(g["spy_close"].iloc[0]); T = float(g["tte_days"].iloc[0])/365.0
    kind = g["type"].iloc[0]
    if (g["type"] != kind).any():   # mixing calls/puts in one fit -> skip
        continue
    fit = calibrate_heston(S, g["strike"].values, g["iv"].values, T, R, Q, kind)
    if fit is None: continue
    rows.append({"date":d,"expiration":e,"n":fit["n"],"rmse_price":fit["rmse_price"],
                 "v0":fit["v0"],"kappa":fit["kappa"],"theta":fit["theta"],
                 "sigma":fit["sigma"],"rho":fit["rho"]})

res = pd.DataFrame(rows)
print("\ncalibrated heston:", len(res))
if len(res):
    print(res[["rmse_price","n"]].describe().round(5).to_string())
    print("\nworst 10:")
    print(res.nlargest(10,"rmse_price")[["date","expiration","n","rmse_price"]].to_string(index=False))
    res.to_csv("audit_out/heston_fit_2024_2025.csv", index=False)
    print("saved: audit_out/heston_fit_2024_2025.csv")

rows: 124417 | days: 165

calibrated heston: 1
       rmse_price     n
count     1.00000   1.0
mean      0.27872  24.0
std           NaN   NaN
min       0.27872  24.0
25%       0.27872  24.0
50%       0.27872  24.0
75%       0.27872  24.0
max       0.27872  24.0

worst 10:
      date expiration  n  rmse_price
2025-04-09 2025-04-22 24    0.278719
saved: audit_out/heston_fit_2024_2025.csv


In [22]:
# src/heston_fast.py
import numpy as np
from scipy.optimize import least_squares
from scipy.stats import norm

_GL_U, _GL_W = np.polynomial.legendre.leggauss(64)
_GL_U = 50 * (_GL_U + 1) / 2
_GL_W = 50 * _GL_W / 2

def _cf(u, S, T, r, q, v0, kappa, theta, sigma, rho):
    u = u.astype(complex); i = 1j
    d = np.sqrt((rho*sigma*i*u - kappa)**2 + sigma**2*(i*u + u**2))
    g = (kappa - rho*sigma*i*u - d) / (kappa - rho*sigma*i*u + d)
    C = (r-q)*i*u*T + (kappa*theta/sigma**2)*((kappa-rho*sigma*i*u-d)*T - 2*np.log((1-g*np.exp(-d*T))/(1-g)))
    D = (kappa-rho*sigma*i*u-d)/sigma**2 * ((1-np.exp(-d*T))/(1-g*np.exp(-d*T)))
    return np.exp(C + D*v0 + i*u*np.log(S))

def heston_price(S, K, T, r, q, v0, kappa, theta, sigma, rho, kind="call"):
    if T <= 0:
        return max(S-K,0) if kind=="call" else max(K-S,0)
    logK = np.log(K); u = _GL_U; w = _GL_W
    phi1 = _cf(u - 1j, S, T, r, q, v0, kappa, theta, sigma, rho)
    P1 = 0.5 + (1/np.pi)*np.sum(w*np.real(np.exp(-1j*u*logK)*phi1/(1j*u*S*np.exp((r-q)*T))))
    phi2 = _cf(u, S, T, r, q, v0, kappa, theta, sigma, rho)
    P2 = 0.5 + (1/np.pi)*np.sum(w*np.real(np.exp(-1j*u*logK)*phi2/(1j*u)))
    call = S*np.exp(-q*T)*P1 - K*np.exp(-r*T)*P2
    return call if kind=="call" else call - S*np.exp(-q*T) + K*np.exp(-r*T)

def _bs(S,K,T,iv,r,q):
    d1 = (np.log(S/K)+(r-q+0.5*iv**2)*T)/(iv*np.sqrt(T)); d2 = d1 - iv*np.sqrt(T)
    return S*np.exp(-q*T)*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def calibrate(S, strikes, ivs, T, r, q, is_call_mask):
    if len(strikes) < 8: return None
    obs = np.array([_bs(S,k,T,iv,r,q) if m else _bs(S,k,T,iv,r,q)-S*np.exp(-q*T)+k*np.exp(-r*T)
                    for k,iv,m in zip(strikes,ivs,is_call_mask)])
    def resid(p):
        v0,kap,th,sig,rho = p
        if not (v0>0 and kap>0 and th>0 and sig>0 and -0.999<rho<0.999):
            return np.ones(len(strikes))*1e3
        try:
            mod = np.array([heston_price(S,k,T,r,q,v0,kap,th,sig,rho,"call" if m else "put")
                            for k,m in zip(strikes,is_call_mask)])
        except Exception:
            return np.ones(len(strikes))*1e3
        return mod - obs
    x0 = [0.04, 2.0, 0.04, 0.5, -0.7]
    bounds = ([1e-3,0.1,1e-3,1e-2,-0.95],[1.0,10.0,1.0,3.0,0.95])
    try:
        r_ = least_squares(resid, x0, bounds=bounds, max_nfev=80)
    except Exception:
        return None
    if not r_.success: return None
    return {**dict(zip(["v0","kappa","theta","sigma","rho"], r_.x)),
            "rmse_price": float(np.sqrt(np.mean(r_.fun**2))), "n": len(strikes)}

In [24]:
# src/heston_eval_v2.py
import os, duckdb, numpy as np, pandas as pd, sys
# sys.path.append(os.path.dirname(__file__))
sys.path.append(r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\src")

# from heston_fast import calibrate

CLEAN_DIR = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\clean"
SPY_FILE  = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\underlying_prices.parquet"
TEMP_DIR  = r"D:\temp\duckdb"
R, Q      = 0.04, 0.013
TTE_MIN, TTE_MAX = 7, 21

con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute(f"SET temp_directory='{TEMP_DIR}'")
con.execute("SET threads=4")
spy = SPY_FILE.replace("\\","/")
srcs = [os.path.join(CLEAN_DIR,f"options_{y}_clean.parquet").replace("\\","/") for y in (2024,2025)]

q = f"""
WITH o AS (
    SELECT CAST(o.date AS DATE) d, CAST(o.expiration AS DATE) e, o.strike, o.type,
           o.implied_volatility iv, o.delta,
           CAST(o.expiration AS DATE) - CAST(o.date AS DATE) AS tte_days
    FROM read_parquet({srcs}) o
    WHERE implied_volatility BETWEEN 0.02 AND 2.0
      AND bid > 0 AND ask > bid
      AND CAST(o.expiration AS DATE) > CAST(o.date AS DATE)
      AND CAST(o.expiration AS DATE) - CAST(o.date AS DATE) BETWEEN {TTE_MIN} AND {TTE_MAX}
      AND EXTRACT(DAY FROM CAST(o.date AS DATE)) <= 5
      AND ABS(o.delta) BETWEEN 0.10 AND 0.90
)
SELECT o.d date, o.e expiration, o.strike, o.type, o.iv,
       s.close spy_close, o.tte_days
FROM o LEFT JOIN '{spy}' s ON o.d = CAST(s.date AS DATE)
WHERE s.close IS NOT NULL
"""
df = con.execute(q).df()
print("rows:", len(df), "| days:", df["date"].nunique())

rows = []
for (d, e), g in df.groupby(["date","expiration"], sort=False):
    if len(g) < 8: continue
    S = float(g["spy_close"].iloc[0]); T = float(g["tte_days"].iloc[0])/365.0
    is_call = (g["type"].values == "call")
    fit = calibrate(S, g["strike"].values, g["iv"].values, T, R, Q, is_call)
    if fit is None: continue
    fit.update({"date":d, "expiration":e}); rows.append(fit)
    if len(rows) % 50 == 0: print(f"  calibrated {len(rows)}...")

res = pd.DataFrame(rows)
print("\ncalibrated:", len(res))
if len(res):
    print(res[["rmse_price","n"]].describe().round(5).to_string())
    print("\nworst 10:")
    print(res.nlargest(10,"rmse_price")[["date","expiration","n","rmse_price"]].to_string(index=False))
    res.to_csv("audit_out/heston_fit_v2.csv", index=False)
    print("saved: audit_out/heston_fit_v2.csv")

rows: 41532 | days: 82
  calibrated 50...
  calibrated 100...
  calibrated 150...
  calibrated 200...
  calibrated 250...
  calibrated 300...
  calibrated 350...
  calibrated 400...
  calibrated 450...
  calibrated 500...

calibrated: 528
       rmse_price          n
count   528.00000  528.00000
mean      0.35985   72.99621
std       0.32986   30.41603
min       0.08195   12.00000
25%       0.21669   51.00000
50%       0.28217   70.00000
75%       0.37336   90.00000
max       2.88966  245.00000

worst 10:
      date expiration   n  rmse_price
2025-04-02 2025-04-09 140    2.889662
2025-04-02 2025-04-11 157    2.879167
2025-04-02 2025-04-10 146    2.851964
2025-04-02 2025-04-17 164    2.707646
2025-04-02 2025-04-14 101    2.705734
2025-04-02 2025-04-16  48    2.144385
2025-04-02 2025-04-15  58    2.056744
2025-05-01 2025-05-16 179    1.431183
2025-03-04 2025-03-14 119    1.229516
2024-08-01 2024-08-12  84    1.137359
saved: audit_out/heston_fit_v2.csv


In [28]:
import os, duckdb, numpy as np, pandas as pd, sys
# sys.path.append(r"src")
# from heston_fast import heston_price
from scipy.stats import norm
from scipy.optimize import brentq

CLEAN_DIR = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\clean"
SPY_FILE  = r"D:\Downloads\options-dataset-hist-main\options-dataset-hist-main\spy\underlying_prices.parquet"
TEMP_DIR  = r"D:\temp\duckdb"
R, Q = 0.04, 0.013
TTE_MIN, TTE_MAX = 7, 21

con = duckdb.connect()
con.execute("SET memory_limit='2GB'"); con.execute(f"SET temp_directory='{TEMP_DIR}'"); con.execute("SET threads=4")
spy = SPY_FILE.replace("\\","/")
srcs = [os.path.join(CLEAN_DIR,f"options_{y}_clean.parquet").replace("\\","/") for y in (2024,2025)]

q = f"""
WITH o AS (
    SELECT CAST(o.date AS DATE) d, CAST(o.expiration AS DATE) e, o.strike, o.type,
           o.implied_volatility iv, o.delta,
           CAST(o.expiration AS DATE) - CAST(o.date AS DATE) AS tte_days
    FROM read_parquet({srcs}) o
    WHERE implied_volatility BETWEEN 0.02 AND 2.0
      AND bid > 0 AND ask > bid
      AND CAST(o.expiration AS DATE) > CAST(o.date AS DATE)
      AND CAST(o.expiration AS DATE) - CAST(o.date AS DATE) BETWEEN {TTE_MIN} AND {TTE_MAX}
      AND EXTRACT(DAY FROM CAST(o.date AS DATE)) <= 5
      AND ABS(o.delta) BETWEEN 0.10 AND 0.90
)
SELECT o.d date, o.e expiration, o.strike, o.type, o.iv,
       s.close spy_close, o.tte_days
FROM o LEFT JOIN '{spy}' s ON o.d = CAST(s.date AS DATE)
WHERE s.close IS NOT NULL
"""
data = con.execute(q).df()
data.to_parquet("audit_out/slice_for_heston.parquet")
print("slice rows:", len(data))

fit = pd.read_csv("audit_out/heston_fit_v2.csv")

def iv_from_price(p, S, K, T, kind):
    if p <= 0 or T <= 0: return np.nan
    intr = max(S-K,0) if kind=="call" else max(K-S,0)
    if p <= intr + 1e-8: return np.nan
    def f(s):
        d1=(np.log(S/K)+(R-Q+0.5*s*s)*T)/(s*np.sqrt(T)); d2=d1-s*np.sqrt(T)
        return (S*np.exp(-Q*T)*norm.cdf(d1)-K*np.exp(-R*T)*norm.cdf(d2)-p) if kind=="call" \
               else (K*np.exp(-R*T)*norm.cdf(-d2)-S*np.exp(-Q*T)*norm.cdf(-d1)-p)
    try: return brentq(f, 1e-3, 5.0, maxiter=80)
    except Exception: return np.nan

rows=[]
for _,r in fit.iterrows():
    g = data[(data["date"]==r["date"]) & (data["expiration"]==r["expiration"])]
    if len(g)==0: continue
    S = float(g["spy_close"].iloc[0]); T = float(g["tte_days"].iloc[0])/365
    iv_hat = [iv_from_price(heston_price(S,k,T,R,Q,r["v0"],r["kappa"],r["theta"],r["sigma"],r["rho"],t),
                            S,k,T,t) for k,t in zip(g["strike"],g["type"])]
    iv_hat = np.array(iv_hat); iv_obs = g["iv"].values
    ok = np.isfinite(iv_hat)
    if ok.sum()<5: continue
    err = iv_hat[ok]-iv_obs[ok]
    rows.append({"date":r["date"],"expiration":r["expiration"],"n":int(ok.sum()),
                 "iv_rmse":float(np.sqrt(np.mean(err**2))),
                 "iv_mae":float(np.mean(np.abs(err)))})

res = pd.DataFrame(rows)
print("smiles compared:", len(res))
print(res[["iv_rmse","iv_mae","n"]].describe().round(5).to_string())
res.to_csv("audit_out/heston_iv_errors.csv", index=False)

slice rows: 41532
smiles compared: 528
         iv_rmse     iv_mae          n
count  528.00000  528.00000  528.00000
mean     0.01346    0.01070   71.67045
std      0.01551    0.01190   29.79164
min      0.00302    0.00251   11.00000
25%      0.00783    0.00606   50.00000
50%      0.00995    0.00790   69.00000
75%      0.01301    0.01075   89.00000
max      0.15783    0.11870  245.00000


In [29]:
pip install streamlit 

  Using cached streamlit-1.66.1-py3-none-any.whl.metadata (10 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pillow-13.0.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached starlette-1.10.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached markupsafe-3.0.4-cp313-cp313-win_amd64.whl.metadata (2.


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
pip install plotly

  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
Using cached plotly-7.1.0-py3-none-any.whl (9.7 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
